# 2SFCA lab: balancing service supply and population demand
Calculate a simple two-step floating catchment area accessibility score from a population-to-provider impedance matrix.

The example uses synthetic Euclidean costs and a binary catchment. A real study should use an appropriate road or transit travel-cost matrix and document its units.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/two-step-fca'))
from two_step_fca import two_step_fca


## 1. Define population and providers

The cost matrix has one row per population location and one column per provider. Supply and demand must be expressed in compatible, documented units.

In [ ]:
population_xy = np.array([[0, 0], [1, 1], [2, 0], [5, 1], [6, 0], [7, 2]])
population = np.array([100, 150, 120, 90, 110, 130], dtype=float)
provider_xy = np.array([[0, 1], [3, 0], [6, 2]])
provider_supply = np.array([20, 35, 25], dtype=float)
costs = np.linalg.norm(population_xy[:, None, :] - provider_xy[None, :, :], axis=2)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(population_xy[:, 0], population_xy[:, 1], s=population / 2, label='Population (size = demand)')
ax.scatter(provider_xy[:, 0], provider_xy[:, 1], marker='s', s=120, label='Provider')
ax.set_title('Population locations and providers'); ax.set_aspect('equal'); ax.legend(); ax.grid(True); plt.show()


## 2. Calculate the two steps

Step one divides each provider's supply by the population demand within its catchment. Step two sums the ratios of reachable providers for each population location.

In [ ]:
threshold = 3.2
ratios, accessibility = two_step_fca(costs, provider_supply, population, max_cost=threshold)
print('Provider supply-to-demand ratios:', np.round(ratios, 4))
print('Accessibility by population location:', np.round(accessibility, 4))
catchment = costs <= threshold
fig, ax = plt.subplots(figsize=(7, 5))
for i in range(len(population_xy)):
    for j in range(len(provider_xy)):
        if catchment[i, j]: ax.plot([population_xy[i, 0], provider_xy[j, 0]], [population_xy[i, 1], provider_xy[j, 1]], alpha=.45)
sc = ax.scatter(population_xy[:, 0], population_xy[:, 1], c=accessibility, s=population / 2, cmap='viridis', edgecolor='black', label='Population')
ax.scatter(provider_xy[:, 0], provider_xy[:, 1], marker='s', s=120, color='red', label='Provider')
ax.set_title(f'2SFCA accessibility, threshold={threshold}'); ax.set_aspect('equal'); ax.legend(); ax.grid(True); fig.colorbar(sc, ax=ax, label='Accessibility score'); plt.show()


## 3. Change the catchment threshold

The binary threshold changes both the population included in each provider's denominator and the providers contributing to each population location's score.

In [ ]:
for threshold in [2.0, 3.2, 5.0]:
    ratios, scores = two_step_fca(costs, provider_supply, population, max_cost=threshold)
    print(f'threshold={threshold}:', np.round(scores, 4))


## 4. Investigate

1. Increase one provider's supply. Which population locations benefit under the catchment model?
2. Change the threshold. Why do provider ratios change as well as accessibility scores?
3. Replace Euclidean distance with network travel time. What assumptions does that introduce?
4. How could distance decay reduce the abrupt boundary effect?
5. Why is an accessibility score not a probability of receiving service?

## 5. Add distance decay (E2SFCA-style)\nThe binary model treats every reachable pair equally. Here we supply travel-time bands and weights. The same weight is used in the provider-demand denominator and in the population accessibility sum, matching the two-stage weighted formulation.

In [ ]:
from two_step_fca import two_step_fca_distance_decay
band_upper_bounds = [1.5, 3.2, 5.0]
band_weights = [1.0, 0.68, 0.22]
decay_ratios, decay_accessibility = two_step_fca_distance_decay(
    costs, provider_supply, population,
    band_upper_bounds=band_upper_bounds, band_weights=band_weights,
)
print('Band upper bounds:', band_upper_bounds)
print('Band weights:', band_weights)
print('Distance-decay provider ratios:', np.round(decay_ratios, 4))
print('Distance-decay accessibility:', np.round(decay_accessibility, 4))
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(population_xy[:, 0], population_xy[:, 1], c=decay_accessibility, s=population / 2, cmap='viridis', edgecolor='black')
ax.scatter(provider_xy[:, 0], provider_xy[:, 1], marker='s', s=120, color='red')
ax.set_title('E2SFCA-style accessibility with supplied bands'); ax.set_aspect('equal'); ax.grid(True)
fig.colorbar(sc, ax=ax, label='Accessibility score'); plt.show()


## 6. Interpret and stress-test the decay schedule\nThese weights are illustrative, not universal. Change the schedule to test whether the relative pattern is robust. A travel-time band for walking is not automatically suitable for driving, public transport or emergency care. The implementation is zonal rather than continuous Gaussian decay, and pairs beyond the last band receive zero weight.\n\n1. Make the outer-band weight smaller. Which population locations change most?\n2. Move a band boundary. Why can both provider ratios and population scores change?\n3. Compare the binary and weighted scores. Which differences are caused by the distance-decay assumption rather than the supply data?